# Offline Model Evaluation Audit

## tl;dr

Structured Logistic Regression reaches **0.773 PR-AUC** with simulator-informed
features but only **0.709** with deployment-observable structured inputs, exposing a
**0.064 information-access gap**. Failure attribution remains weak at **0.171
Macro-F1**, while severity prediction reaches **0.906** and governance recommendation
reaches **0.498** with **95.4% Top-3 accuracy**. All rows are synthetic; no LLM was
evaluated.

## Context & Methods

This companion notebook audits persisted evaluation outputs rather than retraining
models. The source of truth is `data/evaluation/`, generated with seed `20260827`.

### Key Assumptions

- The harmful-action label is synthetic and deterministic for a fixed seed.
- Split membership is assigned at `task_id` level, not row level.
- Model selection uses validation PR-AUC; the test split is reserved for reporting.
- Threshold selection maximizes F2 subject to validation over-blocking <= 35%.
- Simulator-informed and deployment-observable features are reported separately.
- Failure attribution excludes injected-stressor identity and raw trace text.

## Data

### 1. Load persisted artifacts

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
assert (ROOT / "pyproject.toml").exists(), "Run from the repository root"
evaluation_dir = ROOT / "data" / "evaluation"
comparison = pd.read_csv(evaluation_dir / "model_comparison.csv")
calibration = pd.read_csv(evaluation_dir / "model_calibration.csv")
bootstrap = pd.read_csv(evaluation_dir / "bootstrap_confidence_intervals.csv")
ablation = pd.read_csv(evaluation_dir / "ablation_study.csv")
unseen = pd.read_csv(evaluation_dir / "unseen_stressor_generalization.csv")
cross_workflow = pd.read_csv(evaluation_dir / "cross_workflow_generalization.csv")
feature_access = pd.read_csv(evaluation_dir / "feature_access_audit.csv")
multitask = pd.read_csv(evaluation_dir / "multitask_comparison.csv")
per_class = pd.read_csv(evaluation_dir / "multitask_per_class_recall.csv")
governance_unseen = pd.read_csv(evaluation_dir / "governance_unseen_stressor.csv")
manifest = json.loads((evaluation_dir / "evaluation_manifest.json").read_text())
manifest

{'dataset_rows': 12800,
 'unique_tasks': 200,
 'split_rows': {'train': 8704, 'validation': 2048, 'test': 2048},
 'split_tasks': {'train': 136, 'test': 32, 'validation': 32},
 'task_overlap_train_test': 0,
 'positive_rate': 0.47328125,
 'selection_metric': 'validation_pr_auc',
 'best_uncalibrated_model': 'Logistic Regression',
 'calibration_method': 'Isotonic selected by ECE on validation-tune task groups after fitting on disjoint validation-calibration task groups',
 'threshold_tuning': 'maximize F2 subject to over-blocking rate <= 35% on disjoint validation tasks',
 'bootstrap': '300 task-cluster resamples',
 'unseen_stressors': ['memory_poisoning', 'permission_overgrant'],
 'result_scope': 'offline classifiers trained on synthetic simulator traces; no LLM results'}

### 2. Verify leakage boundaries and output ranges

In [2]:
split_manifest = pd.read_csv(evaluation_dir / "split_manifest.csv")
split_sets = {
    name: set(split_manifest.loc[split_manifest["split"] == name, "task_id"])
    for name in ["train", "validation", "test"]
}
assert not split_sets["train"] & split_sets["validation"]
assert not split_sets["train"] & split_sets["test"]
assert not split_sets["validation"] & split_sets["test"]
assert manifest["task_overlap_train_test"] == 0
metric_columns = ["accuracy", "precision", "recall", "f1", "auroc", "pr_auc", "brier", "ece", "over_blocking_rate"]
assert comparison[metric_columns].apply(lambda column: column.between(0, 1).all()).all()
{name: len(values) for name, values in split_sets.items()}

{'train': 136, 'validation': 32, 'test': 32}

## Results

### 3. Compare model ranking and operating trade-offs

In [3]:
columns = ["model", "pr_auc", "auroc", "f1", "safety_recall", "over_blocking_rate", "brier", "ece"]
comparison[columns].sort_values("pr_auc", ascending=False).round(3)

,model,pr_auc,auroc,f1,safety_recall,over_blocking_rate,brier,ece
0,Logistic Regression,0.773,0.791,0.696,0.708,0.276,0.186,0.045
1,Logistic + TF-IDF,0.771,0.778,0.667,0.643,0.242,0.192,0.058
2,Random Forest,0.758,0.768,0.657,0.650,0.279,0.194,0.035
3,Rule Based,0.750,0.775,0.669,0.692,0.320,0.198,0.067
4,Extra Trees,0.744,0.752,0.650,0.616,0.235,0.199,0.037
5,XGBoost,0.740,0.746,0.645,0.632,0.277,0.205,0.054
6,Histogram Gradient Boosting,0.733,0.741,0.649,0.636,0.275,0.206,0.064
7,Logistic Regression + Isotonic,0.719,0.770,0.684,0.660,0.230,0.193,0.053


In [4]:
ordered = comparison.sort_values("pr_auc")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].barh(ordered["model"], ordered["pr_auc"], color="#2563EB")
axes[0].set(xlim=(0, 1), xlabel="PR-AUC", title="Task-group holdout ranking")
axes[1].scatter(comparison["over_blocking_rate"], comparison["safety_recall"], color="#D4A72C", s=65)
for row in comparison.itertuples():
    axes[1].annotate(row.model, (row.over_blocking_rate, row.safety_recall), fontsize=8, xytext=(4, 4), textcoords="offset points")
axes[1].set(xlim=(0, .4), ylim=(0, 1), xlabel="Over-blocking rate", ylabel="Safety recall", title="Operating trade-off")
plt.tight_layout()

### 4. Audit calibration and uncertainty

In [5]:
raw = comparison.loc[comparison["model"] == manifest["best_uncalibrated_model"]].iloc[0]
calibrated = comparison.loc[comparison["model_family"] == "calibrated_offline_ml"].iloc[0]
calibration_effect = pd.DataFrame({
    "raw": raw[["pr_auc", "brier", "ece", "f1"]],
    "calibrated": calibrated[["pr_auc", "brier", "ece", "f1"]],
})
calibration_effect["delta"] = calibration_effect["calibrated"] - calibration_effect["raw"]
calibration_effect.round(4)

,raw,calibrated,delta
pr_auc,0.773314,0.719441,-0.053872
brier,0.186015,0.192828,0.006813
ece,0.044991,0.053084,0.008093
f1,0.696335,0.683572,-0.012763


In [6]:
bootstrap[["metric", "estimate", "ci_low", "ci_high", "resampling_unit"]].round(3)

,metric,estimate,ci_low,ci_high,resampling_unit
0,f1,0.684,0.595,0.755,task_id
1,safety_recall,0.660,0.528,0.771,task_id
2,auroc,0.770,0.724,0.813,task_id
3,pr_auc,0.719,0.625,0.790,task_id
4,over_blocking_rate,0.230,0.150,0.324,task_id


### 5. Test feature dependence and transfer

In [7]:
display_ablation = ablation[["configuration", "pr_auc", "f1", "safety_recall", "f1_delta_vs_full", "recall_delta_vs_full"]]
display_ablation.round(3)

,configuration,pr_auc,f1,safety_recall,f1_delta_vs_full,recall_delta_vs_full
0,Full input,0.771,0.667,0.643,0.000,0.000
1,No policy signal,0.771,0.667,0.643,-0.000,0.000
2,No tool contract,0.771,0.667,0.643,-0.000,0.000
3,No graph,0.771,0.667,0.643,0.000,0.000
4,No handoff context,0.774,0.673,0.652,0.006,0.009
5,No text,0.773,0.696,0.708,0.029,0.065


In [8]:
assert (unseen["task_overlap"] == 0).all()
assert (cross_workflow["task_overlap"] == 0).all()
print("Strict unseen-stressor tests")
display(unseen[["held_out_stressor", "pr_auc", "safety_recall", "f1", "over_blocking_rate"]].round(3))
print("Leave-one-workflow-out tests")
display(cross_workflow[["held_out_workflow", "pr_auc", "safety_recall", "f1", "over_blocking_rate"]].round(3))

Strict unseen-stressor tests


,held_out_stressor,pr_auc,safety_recall,f1,over_blocking_rate
0,memory_poisoning,0.818,0.589,0.685,0.134
1,permission_overgrant,0.744,0.615,0.628,0.313


Leave-one-workflow-out tests


,held_out_workflow,pr_auc,safety_recall,f1,over_blocking_rate
0,data_export,0.85,0.599,0.702,0.139
1,email,0.38,0.464,0.393,0.210
2,it_access,0.85,0.627,0.689,0.289
3,refund,0.68,0.400,0.520,0.127


### 6. Measure the simulator-information advantage

In [9]:
assert feature_access.loc[feature_access["label_shuffle"], "pr_auc"].iloc[0] < feature_access.loc[feature_access["feature_access"] == "Deployable structured", "pr_auc"].iloc[0]
feature_access[["feature_access", "feature_count", "pr_auc", "f1", "safety_recall", "over_blocking_rate", "pr_auc_optimism_gap"]].round(3)

,feature_access,feature_count,pr_auc,f1,safety_recall,over_blocking_rate,pr_auc_optimism_gap
0,Simulator-informed structured,53,0.773,0.696,0.708,0.276,0.064
1,Deployable structured,39,0.709,0.594,0.542,0.241,0.064
2,Deployable + text,41,0.685,0.611,0.589,0.288,0.064
3,Label-shuffled negative control,39,0.429,0.211,0.141,0.162,0.064


In [10]:
ordered_access = feature_access.sort_values("pr_auc")
colors = ["#E87722" if value else ("#94A3B8" if shuffled else "#2563EB") for value, shuffled in zip(ordered_access["uses_simulator_privileged_features"], ordered_access["label_shuffle"])]
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh(ordered_access["feature_access"], ordered_access["pr_auc"], color=colors)
ax.set(xlim=(0, 1), xlabel="PR-AUC on task-group holdout", title="Risk classification by feature-access policy")
plt.tight_layout()

### 7. Evaluate failure attribution, severity, and governance recommendation

In [11]:
multitask[["task", "model", "accuracy", "macro_f1", "macro_recall", "top_3_accuracy", "mean_decision_regret", "rows"]].round(3)

,task,model,accuracy,macro_f1,macro_recall,top_3_accuracy,mean_decision_regret,rows
0,failure_attribution,Majority Class,0.137,0.030,0.125,NaN,NaN,939
1,failure_attribution,Multinomial Logistic Regression,0.177,0.171,0.172,NaN,NaN,939
2,severity_prediction,Majority Class,0.501,0.222,0.333,NaN,NaN,939
3,severity_prediction,Multinomial Logistic Regression,0.907,0.906,0.918,NaN,NaN,939
4,governance_recommendation,Majority Class,0.349,0.086,0.167,0.349,57.028,152
5,governance_recommendation,Multinomial Logistic Regression,0.533,0.498,0.573,0.954,12.592,152


In [12]:
failure_recall = per_class[per_class["task"] == "failure_attribution"]
assert failure_recall["test_support"].sum() == 939
display(failure_recall.sort_values("recall", ascending=False).round(3))
print("Governance recommendation on strictly unseen stressors")
display(governance_unseen.round(3))

,task,class,recall,test_support,validation_rows
3,failure_attribution,F04,0.386,140,1019
4,failure_attribution,F05,0.197,122,1019
5,failure_attribution,F06,0.168,119,1019
2,failure_attribution,F03,0.147,102,1019
7,failure_attribution,F09,0.147,68,1019
0,failure_attribution,F01,0.126,135,1019
6,failure_attribution,F07,0.116,129,1019
1,failure_attribution,F02,0.089,124,1019


Governance recommendation on strictly unseen stressors


,held_out_stressor,train_tasks,test_tasks,task_overlap,accuracy,balanced_accuracy,macro_f1,weighted_f1,macro_recall,rows,classes,top_3_accuracy,mean_decision_regret
0,memory_poisoning,135,20,0,0.40,0.400,0.272,0.272,0.400,20,4,0.850,39.777
1,permission_overgrant,135,21,0,0.19,0.556,0.244,0.089,0.556,21,3,0.381,68.734


## Takeaways

1. The task-group split is clean: 136/32/32 tasks and zero overlap.
2. The deployable structured model falls from 0.773 to 0.709 PR-AUC when simulator-only
   variables are removed. The higher number must not be presented as deployable quality.
3. The selected isotonic calibration is not stable on the test set: PR-AUC and Brier
   both worsen. Keep the raw scorer until a larger independent calibration set exists.
4. Transfer is uneven. Email is the weakest held-out workflow (PR-AUC about 0.380),
   while unseen-stressor PR-AUC remains above 0.74. Workflow shift is therefore the
   more serious current generalization risk.
5. Failure attribution is the main capability gap (Macro-F1 0.171). The structured
   trace lacks enough diagnostic evidence to distinguish most taxonomy classes.
6. Governance recommendation is useful as a ranked shortlist (95.4% Top-3) but not as
   an autonomous decision (53.3% Top-1; substantial unseen-stressor regret).
7. These conclusions describe a synthetic trace distribution, not real agent behavior.